# 01 - Hello, Liveliness!
The point of this is to act as a "Hello, World!" application for a organoid target of a python script using the cl-sdk library from Cortical Labs. It works be breaking up the CL1's MEA into three segments from left to right, acting as a normal, award, and anomal channel groups. Normal and anomaly channel groups are treated as inputs, and the reward grouping are treated as the single output. In this test, normal channel stimulation is delivered with no anomaly input. No reward feedback is issued during this phase — the experiment observes whether the network produces spike activity in response to normal stimulation alone. A positive result is measurable spike output following each stimulation event, establishing baseline stimulus-response behavior before feedback loops are introduced.

#### 01.1 — Setup

Verify that `cl-sdk` is installed and your environment is configured correctly.

In [29]:
import cl
from cl import RecordingView, ChannelSet, StimDesign
from pathlib import Path
import time
import pandas as pd

DATA_DIR = Path("../data")

pd.Series(cl.get_system_attributes()).rename_axis("attribute").to_frame("value")

,value
attribute,
project_id,cl-sdk-project
chip_id,cl-sdk-chip
cell_batch_id,cl-sdk-cell-batch
plugin,{}
system_id,cl1-sdk-000
hostname,Mac.localdomain


In [32]:
last_recording_path = None

with cl.open() as neurons:

    # ── Step 1: Start recording everything ───────────────────────────────
    # Captures raw voltage samples, spike events, and stim timestamps
    # to an HDF5 file for later analysis
    recording = neurons.record(file_location=DATA_DIR)
    print("Recording started — neurons are being observed")

    # ── Step 2: Baseline — observe spontaneous activity ──────────────────
    # Healthy neurons fire spontaneously without any stimulus.
    # This 5-second window tells you what the network looks like at rest.
    print("Observing baseline spontaneous activity for 5 seconds...")
    last_ts = neurons.timestamp()
    for _ in range(5):
        time.sleep(1)
        current_ts = neurons.timestamp()
        delta = current_ts - last_ts
        neurons.read(delta, last_ts)
        neurons._read_spikes(delta, last_ts)
        last_ts = current_ts

    # ── Step 3: Define WHERE to stimulate ────────────────────────────────
    # The CL1 has 59 electrodes on the multielectrode array.
    # We'll stimulate a small cluster in the center of the array.
    stim_channels = ChannelSet(8, 9, 10)

    # ── Step 4: Define WHAT to send ──────────────────────────────────────
    # Biphasic pulse: negative leading edge (better for neural health),
    # 160 microsecond pulse width, ±1.0 microamp current.
    # Charge-balanced — equal positive and negative charge so you don't
    # damage the neurons or the electrodes over repeated stimulation.
    stim = StimDesign(
        160, -1.0,
        160,  1.0,
    )

    # ── Step 5: Deliver the stimulus ─────────────────────────────────────
    print("Delivering stimulus pulse...")
    neurons.stim(stim_channels, stim)

    # ── Step 6: Observe the response window ──────────────────────────────
    # Neural responses to a stimulus typically occur within
    # 1–50 milliseconds. Record for 5 seconds to capture
    # any downstream network propagation.
    print("Recording post-stimulus response window...")
    last_ts = neurons.timestamp()
    for _ in range(5):
        time.sleep(1)
        current_ts = neurons.timestamp()
        delta = current_ts - last_ts
        neurons.read(delta, last_ts)
        neurons._read_spikes(delta, last_ts)
        last_ts = current_ts

    # ── Step 7: Stop and save ─────────────────────────────────────────────
    recording.stop()
    last_recording_path = recording.file["path"]
    print(f"Recording complete — saved to {last_recording_path}")

Recording started — neurons are being observed
Observing baseline spontaneous activity for 5 seconds...
Delivering stimulus pulse...
Recording post-stimulus response window...
Recording complete — saved to /Users/kirklandyuknis/Desktop/learning-organoid-programming/data/2026-03-14_17-51-15.604+00-00_recording.h5


In [33]:
recording = RecordingView(last_recording_path)

# Total spikes detected across the entire recording
total_spikes = len(recording.spikes)
print(f"Total spikes detected: {total_spikes}")

# Raw voltage across all 59 channels
# Shape: (duration_frames × channel_count)
print(f"Sample array shape: {recording.samples.shape}")

# When stimuli were delivered
print(f"Stim events: {recording.stims}")

recording.close()

Total spikes detected: 693
Sample array shape: (np.int64(250980), np.int64(64))
Stim events: /stims (Table(np.int64(0),)) np.str_('')
